In [16]:
# Imports and base setup
import os
import re
import json
import hashlib
from typing import Dict, List, Tuple

# Must be set BEFORE importing huggingface_hub/transformers
os.environ["HF_HUB_DISABLE_XET"] = "1"

# Patch SSL verification BEFORE importing transformers/huggingface
import urllib3
urllib3.disable_warnings()

# Patch HTTPX to disable SSL verification
try:
    import httpx
    httpx._verify_disabled = True
except ImportError:
    pass

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from sentence_transformers import CrossEncoder
from huggingface_hub import login

In [17]:
import faiss
import numpy as np
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_groq import ChatGroq
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

from pathlib import Path
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
from dotenv import load_dotenv


env_path = Path.cwd() / ".env"
if not env_path.exists():
    env_path = Path.cwd().parent / ".env"
load_dotenv(env_path)  # Load environment variables from workspace root .env file


True

In [18]:
# Configure model names and initialize the LLM
embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
llm_model_name = "qwen/qwen3-32b"

hf_token = os.getenv("HF_TOKEN", "").strip().strip('"').strip("'")
if hf_token:
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = hf_token

llm = ChatGroq(
    model=llm_model_name,
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
 )

print(f"Embedding model: {embedding_model_name}")
print(f"LLM model: {llm_model_name}")

Embedding model: sentence-transformers/all-MiniLM-L6-v2
LLM model: qwen/qwen3-32b


In [19]:
# Load source text
candidates = [
    Path.cwd() / "data.txt",
    Path.cwd().parent / "data.txt",
]

data_path = next((p for p in candidates if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("data.txt not found in current directory or parent directory.")

text_data = data_path.read_text(encoding="utf-8")
print(f"Loaded {len(text_data)} characters from {data_path}")

Loaded 50293 characters from c:\projects\learn-rag\vectorDB\data.txt


In [35]:
# Split text into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = splitter.split_text(text_data)
print(f"Created {len(chunks)} chunks")
print(chunks[0][:250])

Created 153 chunks
The Indian Premier League (IPL) is a professional Twenty20 (T20) cricket league in India, organised by the Board of Control for Cricket in India (BCCI).[1] Founded in 2007, it features ten city-based franchise teams.[2] The IPL is the most popular an


In [43]:
chunks

['The Indian Premier League (IPL) is a professional Twenty20 (T20) cricket league in India, organised by the Board of Control for Cricket in India (BCCI).[1] Founded in 2007, it features ten city-based franchise teams.[2] The IPL is the most popular and richest cricket league in the world and the 11th richest sporting league in the world by revenue. It is held annually between March and May',
 '. It is held annually between March and May. It has an exclusive window in the Future Tours Programme of the International Cricket Council, resulting in fewer international tours occurring during the seasons.[3] It is also the most viewed Indian sports event, per the Broadcast Audience Research Council.[4][5]',
 'In 2010, the IPL became the first sporting event to broadcast live on YouTube.[6][7] In 2014, it ranked sixth in attendance among all sports leagues.[8] Inspired by the success of the IPL, other Indian sports leagues have been established.[a][11][12] The IPL is the second-richest sports

In [81]:
from langchain_text_splitters import SpacyTextSplitter

spacy_splitter = SpacyTextSplitter(
    pipeline="sentencizer",
    chunk_size=500,
    chunk_overlap=80,
)
spacy_chunks = spacy_splitter.split_text(text_data)

Created a chunk of size 877, which is longer than the specified 500
Created a chunk of size 917, which is longer than the specified 500
Created a chunk of size 569, which is longer than the specified 500
Created a chunk of size 535, which is longer than the specified 500
Created a chunk of size 2017, which is longer than the specified 500
Created a chunk of size 1630, which is longer than the specified 500
Created a chunk of size 1068, which is longer than the specified 500
Created a chunk of size 794, which is longer than the specified 500
Created a chunk of size 2105, which is longer than the specified 500
Created a chunk of size 4679, which is longer than the specified 500
Created a chunk of size 1771, which is longer than the specified 500
Created a chunk of size 2464, which is longer than the specified 500
Created a chunk of size 1725, which is longer than the specified 500
Created a chunk of size 1685, which is longer than the specified 500
Created a chunk of size 1127, which is 

In [82]:
spacy_chunks

['The Indian Premier League (IPL) is a professional Twenty20 (T20) cricket league in India, organised by the Board of Control for Cricket in India (BCCI).[1] Founded in 2007, it features ten city-based franchise teams.[2] The IPL is the most popular and richest cricket league in the world and the 11th richest sporting league in the world by revenue.\n\nIt is held annually between March and May.',
 'It has an exclusive window in the Future Tours Programme of the International Cricket Council, resulting in fewer international tours occurring during the seasons.[3] It is also the most viewed Indian sports event, per the Broadcast Audience Research Council.[4][5]\n\nIn 2010, the IPL became the first sporting event to broadcast live on YouTube.[6][7] In 2014, it ranked sixth in attendance among all sports leagues.[8] Inspired by the success of the IPL, other Indian sports leagues have been established.[a][11][12] The IPL is the second-richest sports league in the world by per-match value, a

In [83]:
# Load a very lightweight Hugging Face embedding model
embedder = SentenceTransformer(embedding_model_name, device="cpu")

print(f"Loaded embedder: {embedding_model_name}")
print(f"Embedding dimension: {embedder.get_embedding_dimension()}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1944.22it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded embedder: sentence-transformers/all-MiniLM-L6-v2
Embedding dimension: 384


In [84]:
from langchain_core.embeddings import Embeddings

class SentenceTransformerEmbeddings(Embeddings):
    def __init__(self, model):
        self.model = model

    def embed_documents(self, texts):
        return self.model.encode(texts, convert_to_numpy=True).tolist()

    def embed_query(self, text):
        return self.model.encode([text], convert_to_numpy=True)[0].tolist()

embedding_adapter = SentenceTransformerEmbeddings(embedder)


In [85]:
vectors = embedding_adapter.embed_documents(spacy_chunks)
print(f"Chunk vectors shape: {len(vectors)} x {len(vectors[0])}")

Chunk vectors shape: 70 x 384


In [86]:
vectors

[[-0.022484909743070602,
  -0.01445852406322956,
  -0.01014732290059328,
  -0.012542959302663803,
  0.005511323921382427,
  0.019280878826975822,
  0.015570444986224174,
  0.03802909329533577,
  0.12313149869441986,
  0.05748539790511131,
  -0.04125033691525459,
  -0.0006190711865201592,
  0.029812734574079514,
  0.05719967186450958,
  0.05658046528697014,
  -0.036760542541742325,
  -0.028455521911382675,
  -0.0692044198513031,
  -0.024707604199647903,
  -0.14173856377601624,
  0.02092605084180832,
  -0.0036825540009886026,
  -0.03700140118598938,
  -0.002437481191009283,
  -0.010835651308298111,
  -0.024455100297927856,
  -0.025875644758343697,
  0.06072200834751129,
  -0.03335914388298988,
  -0.04299633577466011,
  0.02201019786298275,
  0.06737248599529266,
  0.03851205110549927,
  0.07595569640398026,
  -0.10321763902902603,
  -0.0789681226015091,
  -0.03454836830496788,
  0.036479879170656204,
  0.0429966039955616,
  -0.018221329897642136,
  0.04075569659471512,
  -0.0950232669711

In [87]:
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

embedding_dim = embedder.get_embedding_dimension()
index = faiss.IndexFlatL2(embedding_dim)

vector_store = FAISS(
    embedding_function=embedding_adapter,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

print(f"Vector store initialized with dimension: {embedding_dim}")

Vector store initialized with dimension: 384


There are 2 approaches like we can precompute our embeddings and send to the VectorDB or we can directly send our details to the vectorDB it will use Sentence transformer and automatically embedd those documents. 

In [91]:
from uuid_utils import uuid4

uuids = [str(uuid4()) for _ in range(len(vectors))]
text_embeddings = list(zip(spacy_chunks, vectors))

vector_store.add_embeddings(text_embeddings=text_embeddings, ids=uuids)
print(f"Added {len(uuids)} embeddings to FAISS")

Added 70 embeddings to FAISS


In [93]:
# Save FAISS vector store to disk and reload for inference
store_dir = Path.cwd() / "faiss_store"
store_dir.mkdir(parents=True, exist_ok=True)

vector_store.save_local(folder_path=str(store_dir))
print(f"Saved vector store to: {store_dir}")

loaded_vector_store = FAISS.load_local(
    folder_path=str(store_dir),
    embeddings=embedding_adapter,
    allow_dangerous_deserialization=True,
 )

query = "what was the name of Deccan chargers after it was banned ?"
results = loaded_vector_store.similarity_search(query, k=4)

print("Top 3 retrieved chunks from loaded store:")
for i, doc in enumerate(results, start=1):
    print(f"\nResult {i}:\n{doc.page_content[:500]}")

Saved vector store to: c:\projects\learn-rag\vectorDB\faiss_store
Top 3 retrieved chunks from loaded store:

Result 1:
This change in playing conditions was trialled during the 2023–24 Syed Mushtaq Ali Trophy, India's domestic T20 tournament.[70]
Teams
Indian Premier League is located in IndiaMIMIKKRKKRPBKSPBKSRCBRCBRRRRDCDCSRHSRHCSKCSKGTGTLSGLSG
All 10 IPL teams in the cities they are based in
The IPL began in 2008 IPL with eight teams.

Over the years, the league saw several team changes.

Deccan Chargers, were terminated in 2012 due to financial issues and were replaced by Sunrisers Hyderabad in 2013.

Result 2:
This change in playing conditions was trialled during the 2023–24 Syed Mushtaq Ali Trophy, India's domestic T20 tournament.[70]
Teams
Indian Premier League is located in IndiaMIMIKKRKKRPBKSPBKSRCBRCBRRRRDCDCSRHSRHCSKCSKGTGTLSGLSG
All 10 IPL teams in the cities they are based in
The IPL began in 2008 IPL with eight teams.

Over the years, the league saw several team changes.
